# IMDb Movie Reviews Sentiment Analysis Exploration
This notebook demonstrates dataset exploration, tokenization, basic model loading, inference, and model explainability (XAI) using Captum.

### 1. Setup & Device Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

### 2. Loading the IMDb Dataset

In [ ]:
train_df = pd.read_csv("../data/Train.csv")
test_df = pd.read_csv("../data/Test.csv")
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# Display first few rows
train_df.head()

### 3. Data Distribution Analysis

In [ ]:
# Plot labels distribution
label_counts = train_df["label"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(["Negative (0)", "Positive (1)"], label_counts.values, color=["#dc3545", "#28a745"])
plt.title("Label Distribution in Training Set")
plt.ylabel("Count")
plt.show()

### 4. Tokenization Walkthrough

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
sample_text = "This movie was spectacular and highly entertaining!"

inputs = tokenizer(sample_text, truncation=True, padding="max_length", max_length=15, return_tensors="pt")
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Input IDs:", inputs["input_ids"][0])
print("Attention Mask:", inputs["attention_mask"][0])

### 5. Prediction with Local Model Checkpoint

In [ ]:
model_path = "../models/best_model"
if os.path.exists(model_path):
    model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    
    inputs = tokenizer(sample_text, truncation=True, padding="max_length", max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).squeeze()
    
    pred_idx = torch.argmax(probs).item()
    labels = ["Negative", "Positive"]
    print(f"Review: '{sample_text}'")
    print(f"Prediction: {labels[pred_idx]} (Confidence: {probs[pred_idx]:.4f})")
else:
    print("Best model not trained yet. Run python src/train.py first!")

### 6. Explainability using Captum (Integrated Gradients)

In [ ]:
if os.path.exists(model_path):
    from captum.attr import LayerIntegratedGradients
    
    def forward_func(input_ids, attention_mask=None):
        return model(input_ids=input_ids, attention_mask=attention_mask).logits
        
    lig = LayerIntegratedGradients(forward_func, model.distilbert.embeddings)
    
    # Get inputs
    inputs_clean = tokenizer(sample_text, truncation=True, max_length=128, return_tensors="pt").to(device)
    input_ids = inputs_clean["input_ids"]
    attention_mask = inputs_clean["attention_mask"]
    
    # Baseline reference input
    ref_input_ids = torch.zeros_like(input_ids)
    ref_input_ids[0, 0] = tokenizer.cls_token_id
    ref_input_ids[0, -1] = tokenizer.sep_token_id
    
    # Run attribution
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=ref_input_ids,
        additional_forward_args=(attention_mask,),
        target=pred_idx,
        return_convergence_delta=True
    )
    
    attributions_sum = attributions.sum(dim=-1).squeeze(0)
    attributions_sum = attributions_sum / torch.norm(attributions_sum)
    
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    for token, score in zip(tokens, attributions_sum.cpu().tolist()):
        if token not in ["[CLS]", "[SEP]", "[PAD]"]:
            print(f"{token:15} : {score:+.4f}")